In [ ]:
import kagglehub

In [ ]:
# Download latest version
path = kagglehub.dataset_download("dylanjcastillo/7k-books-with-metadata")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd

In [ ]:
books = pd.read_csv(f"{path}/books.csv")

In [ ]:
books

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
ax = plt.axes()
sns.heatmap(books.isna().transpose(), cbar=False, ax=ax)

plt.xlabel('Columns')
plt.ylabel('Missing Values')

plt.show()

In [ ]:
import numpy as np

books["missing_description"] = np.where(books["description"].isna(), 1, 0)
books["age_of_book"] = 2026 - books["published_year"]

In [ ]:
col = ["num_pages", "age_of_book", "missing_description", "average_rating"]
corr_matrix = books[col].corr(method = "spearman")

plt.figure(figsize=(8,6))
heatmap = sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            cbar_kws={"label": "spearman correlation"})
heatmap.set_title("Correlation Heatmap")
plt.show()

In [ ]:
books[books["num_pages"].isna() 
      | books["published_year"].isna()
      | books["description"].isna()
      | books["average_rating"].isna()]

In [ ]:
clear_books = books[~books["num_pages"].isna() 
      & ~books["description"].isna()
      & ~books["average_rating"].isna()
      & ~books["published_year"].isna()]

In [ ]:
clear_books["categories"].value_counts().reset_index().sort_values("count", ascending=False)

In [ ]:
clear_books["description_length"] = clear_books["description"].apply(lambda x: len(str(x).split()))

In [ ]:
clear_books

In [ ]:
clear_books.loc[clear_books["description_length"].between(25,34), "description"]

In [ ]:
clear_books = clear_books[clear_books["description_length"] >= 25]
clear_books

In [ ]:
clear_books["title and subtitle"] = np.where(clear_books["subtitle"].isna(), clear_books["title"],
                            clear_books["title"] + ": " + clear_books["subtitle"])

In [ ]:
clear_books

In [ ]:
clear_books["tagged_description"] = clear_books[["isbn13", "description"]].astype(str).agg(" ".join, axis=1)
clear_books

In [ ]:
clear_books.drop(["subtitle", "missing_description", "age_of_book", "description_length"], axis=1).to_csv("books_cleaned.csv", index=False)
